[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20I%20-%20The%20Port%20%26%20Container%20Terminal%20%28Problems%201-46%29/C.%20Core%20Yard%20%26%20Land-Side%20Operations%20%28The%20Heart%20of%20the%20Terminal%29/26.%20The%20Gate%20Automation%20%26%20Damage%20Detection%20Problem/P26-Tier-2_executed.ipynb) [![Open in SageMaker](https://img.shields.io/badge/Open%20in-SageMaker-orange?logo=amazon-aws)](https://aws.amazon.com/sagemaker/) [![Open in Azure Notebooks](https://img.shields.io/badge/Open%20in-Azure%20Notebooks-blue?logo=microsoft-azure)](https://notebooks.azure.com/)

---

# 26. The Gate Automation & Damage Detection Problem

## Tier 2: A Priority-Based Greedy Heuristic

### Goal

Implement and examine a priority-based greedy algorithm for allocating a limited hourly monitoring budget across gate sensors. At each hour, the algorithm ranks sensors using their health, failure history, the demand at their gate, environmental stress, and sensor criticality. It then monitors the highest-priority sensors that fit within the budget.

The method is a heuristic: it makes a locally informed allocation at each hour and does not guarantee a globally optimal schedule. The simulation also models sensor-state changes and gate availability over time, so the priorities can change as the simulated system evolves.

### Inputs, outputs, and scope

**Inputs** are the gates and sensors, hourly demand patterns, hourly monitoring budget, sensor monitoring costs, and initial sensor conditions. **Outputs** include the hourly monitoring schedule, gate-level availability, sensor-health histories, system reliability, and budget utilization.

This is a teaching simulation, not a calibrated reliability model. Monitoring improves a sensor's health by a fixed amount; an unmonitored sensor degrades according to its environmental factor. Random failures are possible only when health is below a specified threshold. A gate is operational when its average sensor health meets the model's threshold. The model does not represent repair crews, sensor redundancy, correlated failures, monitoring setup costs, or uncertainty in the input demand patterns.

### Solution sequence

1. Define the data structures and the 48-hour example.
2. Calculate sensor priorities and use a priority queue to select sensors within budget.
3. Update sensor and gate states and record results for each hour.
4. Analyze monitoring patterns, visualize the simulation, and examine computational scaling.

The numerical results depend on random draws in the simulation. No random seed is specified, so separate executions need not produce identical histories.

## Model and algorithm

### Priority score

A sensor's priority is a weighted sum of five normalized components. A higher score means that the greedy rule considers the sensor more critical to monitor at the current hour:

$$
P_{g,s,t} = w_h H_{g,s,t} + w_f F_{g,s,t} + w_d D_{g,t} + w_e E_{g,s,t} + w_c C_s.
$$

Here, $g$ identifies a gate, $s$ a sensor at that gate, and $t$ an hour. The health component is $H_{g,s,t} = 1 - h_{g,s,t}$, where $h_{g,s,t}$ is the sensor's health index. The failure-history component is $F_{g,s,t} = \min(f_{g,s,t}/10, 1)$, where $f_{g,s,t}$ is its recorded failure count. The demand component is $D_{g,t} = d_{g,t}/150$, where $d_{g,t}$ is the gate's demand in the example. The environmental component is $E_{g,s,t} = (e_{g,s} - 0.5)/1.5$, where $e_{g,s}$ is the environmental stress factor. The criticality component is $C_s = c_s/3$, where the model assigns criticality values of 3 to a safety beam, 2 to a photo eye, and 1 to a ground loop.

The weights are $w_h=0.30$, $w_f=0.25$, $w_d=0.20$, $w_e=0.15$, and $w_c=0.10$. They sum to 1 and define the relative influence of the components; they are modeling choices, not values estimated from data. In the code, `calculate_priority_score` computes this score using `priority_weights` and `sensor_criticality`.

### Budget-constrained greedy selection

Let $B$ be the hourly budget and $c_s$ the hourly cost of monitoring sensor type $s$. For each hour, the algorithm inserts all sensors into a max-priority queue, repeatedly removes the sensor with the highest score, and selects it if its monitoring cost fits the remaining budget. The remaining budget is reduced by that cost. A sensor that does not fit is skipped, and the algorithm continues considering the other queued sensors.

This is a greedy, priority-ordered selection rule. It is not an optimization guarantee: the score ranks individual sensors, while the budget determines which ranked sensors can be selected. In the code, `select_sensors_to_monitor` implements this rule, and `PrioritySensor.__lt__` makes Python's min-heap return the highest-priority item first. Equal scores do not have a separately specified tie-breaking policy.

### State updates and reported reliability

For a monitored sensor, health increases by 0.01 up to 1.0, and the hourly failure probability is $0.005e_{g,s}$. For an unmonitored sensor, health decreases by $0.02e_{g,s}$ down to 0, and the hourly failure probability is $0.02e_{g,s}$. A failure is counted only when the random draw is below that probability and the updated health is below 0.3; the failure count then increases by one and health falls by a further 0.1, down to 0. These rules are implemented in `update_sensor_states`.

For each gate, the code calculates the mean health of its sensors. The gate is operational if this mean is at least 0.2. System reliability for hour $t$ is the fraction of gates operational, expressed as a percentage:

$$
R_t = 100\,\frac{\text{number of operational gates at hour }t}{\text{number of gates}}.
$$

Thus, this reported reliability is a gate-availability measure in the simulation. It is not a probability estimated from repeated independent runs. `update_gate_states` applies the gate rule, and `run_simulation` records $R_t$ and hourly budget utilization.

### Computational cost

Let $G$ be the number of gates, $S$ the number of sensors, and $T$ the number of simulated hours. In each hour, scoring and inserting the sensors into the heap takes $O(S\log S)$ time, and removing sensors from the heap takes at most another $O(S\log S)$. The gate update scans the sensor list for each gate, taking $O(GS)$ time in this implementation. The resulting time bound is $O\bigl(T(S\log S + GS)\bigr)$, and the recorded hourly histories use $O(TS)$ space. These bounds describe the operations in this implementation; they do not imply a particular wall-clock runtime.

In [ ]:
# Import the libraries used by the simulation, analysis, and visualizations.
import heapq
import random
from collections import defaultdict
from dataclasses import dataclass
from typing import Dict, List, Tuple

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

plt.style.use('seaborn-v0_8')
sns.set_palette('husl')

In [ ]:
# These data classes keep the inputs and evolving state explicit.
@dataclass
class SensorState:
    """Current condition and monitoring history of one sensor."""

    gate_id: int
    sensor_type: str
    health_index: float = 1.0
    failure_count: int = 0
    last_maintenance_hour: int = 0
    monitoring_hours: int = 0
    environmental_factor: float = 1.0


@dataclass
class GateState:
    """Current operational state and traffic totals for one gate."""

    gate_id: int
    demand_level: float = 1.0
    is_operational: bool = True
    uptime_percentage: float = 100.0
    total_vehicles_processed: int = 0


@dataclass
class PrioritySensor:
    """Sensor record ordered so a heap returns the highest priority first."""

    priority: float
    gate_id: int
    sensor_type: str
    timestamp: int

    def __lt__(self, other):
        # Reversing the comparison adapts Python's min-heap to this max-priority rule.
        return self.priority > other.priority


@dataclass
class HeuristicInstance:
    """Complete input instance for the priority-based simulation."""

    gates: List[GateState]
    sensors: List[SensorState]
    time_periods: int
    monitoring_budget_per_hour: float
    sensor_costs: Dict[str, float]
    gate_downtime_costs: Dict[int, float]
    demand_patterns: Dict[int, List[float]]

## Create the example instance

The instance has four gates, three sensor types at each gate, and 48 hourly periods. Demand patterns repeat every 24 hours, with higher demand in the morning and evening peak windows. The hourly budget and sensor costs determine how many sensors the greedy selection can afford; there is no separate fixed limit on the number selected.

In [ ]:
def create_heuristic_example():
    """Build the four-gate, 48-hour example used throughout the notebook."""
    sensor_costs = {
        'photo_eye': 50,
        'ground_loop': 60,
        'safety_beam': 70,
    }

    gate_downtime_costs = {
        1: 15000,
        2: 12000,
        3: 10000,
        4: 8000,
    }

    demand_patterns = {}
    for gate_id in range(1, 5):
        base_demand = 150 - (gate_id - 1) * 25
        pattern = []

        for hour in range(48):
            hour_of_day = hour % 24
            if 8 <= hour_of_day <= 10 or 17 <= hour_of_day <= 19:
                demand_factor = 1.8
            elif 6 <= hour_of_day <= 11 or 16 <= hour_of_day <= 21:
                demand_factor = 1.3
            else:
                demand_factor = 0.7

            pattern.append(base_demand * demand_factor)

        demand_patterns[gate_id] = pattern

    gates = [
        GateState(
            gate_id=gate_id,
            demand_level=demand_patterns[gate_id][0] / 100,
        )
        for gate_id in range(1, 5)
    ]

    sensor_configs = [
        # (gate_id, sensor_type, initial_health, failure_count, environmental_factor)
        (1, 'photo_eye', 0.95, 5, 1.2),
        (1, 'ground_loop', 0.85, 2, 1.0),
        (1, 'safety_beam', 0.90, 1, 1.1),
        (2, 'photo_eye', 0.88, 3, 1.0),
        (2, 'ground_loop', 0.92, 1, 1.3),
        (2, 'safety_beam', 0.87, 4, 1.1),
        (3, 'photo_eye', 0.93, 1, 0.9),
        (3, 'ground_loop', 0.89, 2, 1.0),
        (3, 'safety_beam', 0.91, 1, 0.8),
        (4, 'photo_eye', 0.94, 0, 0.8),
        (4, 'ground_loop', 0.96, 0, 0.9),
        (4, 'safety_beam', 0.95, 1, 0.7),
    ]

    sensors = [
        SensorState(
            gate_id=gate_id,
            sensor_type=sensor_type,
            health_index=health,
            failure_count=failures,
            environmental_factor=environmental_factor,
        )
        for (
            gate_id,
            sensor_type,
            health,
            failures,
            environmental_factor,
        ) in sensor_configs
    ]

    return HeuristicInstance(
        gates=gates,
        sensors=sensors,
        time_periods=48,
        monitoring_budget_per_hour=200,
        sensor_costs=sensor_costs,
        gate_downtime_costs=gate_downtime_costs,
        demand_patterns=demand_patterns,
    )


heuristic_instance = create_heuristic_example()
print('Heuristic instance created:')
print(f'- {len(heuristic_instance.gates)} gates')
print(f'- {len(heuristic_instance.sensors)} sensors')
print(f'- {heuristic_instance.time_periods} hourly periods')
print(
    f'- Monitoring budget: '
    f'${heuristic_instance.monitoring_budget_per_hour:.0f} per hour'
)

print('\nInitial sensor conditions (health index):')
for sensor in heuristic_instance.sensors[:4]:
    print(
        f'Gate {sensor.gate_id} - {sensor.sensor_type}: '
        f'{sensor.health_index:.2f}'
    )

## Implement the priority-based monitoring algorithm

The class below keeps the solution steps visible: score each sensor, select a budget-feasible set from the priority queue, update sensor and gate states, and record the hourly results. Each method uses the instance supplied to the algorithm rather than relying on hidden global input data.

In [ ]:
class PriorityBasedSensorMonitoring:
    """Greedy, priority-based allocation of hourly sensor monitoring."""

    def __init__(self, instance: HeuristicInstance):
        self.instance = instance
        self.priority_weights = {
            'health': 0.3,
            'failure_history': 0.25,
            'demand': 0.2,
            'environmental': 0.15,
            'criticality': 0.1,
        }
        self.sensor_criticality = {
            'safety_beam': 3.0,
            'photo_eye': 2.0,
            'ground_loop': 1.0,
        }
        self.results = {
            'monitoring_schedule': [],
            'system_reliability': [],
            'budget_utilization': [],
            'gate_availabilities': [],
            'sensor_healths': [],
        }

    def calculate_priority_score(
        self,
        sensor: SensorState,
        gate: GateState,
        hour: int,
    ) -> float:
        """Return the weighted priority score for a sensor at one hour."""
        health_score = (
            (1.0 - sensor.health_index) * self.priority_weights['health']
        )
        failure_score = (
            min(sensor.failure_count / 10.0, 1.0)
            * self.priority_weights['failure_history']
        )
        current_demand = (
            self.instance.demand_patterns[gate.gate_id][hour] / 150.0
        )
        demand_score = current_demand * self.priority_weights['demand']
        environmental_score = (
            (sensor.environmental_factor - 0.5) / 1.5
            * self.priority_weights['environmental']
        )
        criticality_score = (
            self.sensor_criticality[sensor.sensor_type] / 3.0
            * self.priority_weights['criticality']
        )

        return (
            health_score
            + failure_score
            + demand_score
            + environmental_score
            + criticality_score
        )

    def select_sensors_to_monitor(self, hour: int) -> List[Tuple[int, str]]:
        """Choose high-priority sensors while respecting the hourly budget."""
        priority_queue = []

        for sensor in self.instance.sensors:
            gate = self.instance.gates[sensor.gate_id - 1]
            priority = self.calculate_priority_score(sensor, gate, hour)
            heapq.heappush(
                priority_queue,
                PrioritySensor(
                    priority=priority,
                    gate_id=sensor.gate_id,
                    sensor_type=sensor.sensor_type,
                    timestamp=hour,
                ),
            )

        selected_sensors = []
        remaining_budget = self.instance.monitoring_budget_per_hour

        while priority_queue and remaining_budget > 0:
            priority_sensor = heapq.heappop(priority_queue)
            sensor_cost = self.instance.sensor_costs[
                priority_sensor.sensor_type
            ]

            if sensor_cost <= remaining_budget:
                selected_sensors.append(
                    (priority_sensor.gate_id, priority_sensor.sensor_type)
                )
                remaining_budget -= sensor_cost

        return selected_sensors

    def update_sensor_states(
        self,
        monitored_sensors: List[Tuple[int, str]],
        hour: int,
    ) -> None:
        """Apply the model's health, monitoring, and random-failure rules."""
        monitored_set = set(monitored_sensors)

        for sensor in self.instance.sensors:
            sensor_key = (sensor.gate_id, sensor.sensor_type)
            is_monitored = sensor_key in monitored_set

            if is_monitored:
                sensor.monitoring_hours += 1
                sensor.health_index = min(1.0, sensor.health_index + 0.01)
                failure_probability = 0.005 * sensor.environmental_factor
            else:
                degradation = 0.02 * sensor.environmental_factor
                sensor.health_index = max(
                    0.0,
                    sensor.health_index - degradation,
                )
                failure_probability = 0.02 * sensor.environmental_factor

            if (
                random.random() < failure_probability
                and sensor.health_index < 0.3
            ):
                sensor.failure_count += 1
                sensor.health_index = max(0.0, sensor.health_index - 0.1)

    def update_gate_states(self, hour: int) -> None:
        """Update gate demand, availability, and processed-vehicle totals."""
        for gate in self.instance.gates:
            gate.demand_level = (
                self.instance.demand_patterns[gate.gate_id][hour] / 100.0
            )
            gate_sensors = [
                sensor
                for sensor in self.instance.sensors
                if sensor.gate_id == gate.gate_id
            ]
            average_health = np.mean(
                [sensor.health_index for sensor in gate_sensors]
            )
            gate.is_operational = average_health >= 0.2

            if gate.is_operational:
                vehicles_processed = int(
                    self.instance.demand_patterns[gate.gate_id][hour]
                )
                gate.total_vehicles_processed += vehicles_processed

    def run_simulation(self) -> Dict:
        """Run the hourly selection and state-update sequence."""
        print('Running the Priority-Based Sensor Monitoring simulation...')

        for hour in range(self.instance.time_periods):
            monitored_sensors = self.select_sensors_to_monitor(hour)
            self.update_sensor_states(monitored_sensors, hour)
            self.update_gate_states(hour)

            self.results['monitoring_schedule'].append(monitored_sensors)

            operational_gates = sum(
                gate.is_operational for gate in self.instance.gates
            )
            reliability = (
                operational_gates / len(self.instance.gates) * 100
            )
            self.results['system_reliability'].append(reliability)

            monitoring_cost = sum(
                self.instance.sensor_costs[sensor_type]
                for _, sensor_type in monitored_sensors
            )
            budget_utilization = (
                monitoring_cost
                / self.instance.monitoring_budget_per_hour
                * 100
            )
            self.results['budget_utilization'].append(budget_utilization)

            gate_availability = {
                gate.gate_id: 100 if gate.is_operational else 0
                for gate in self.instance.gates
            }
            self.results['gate_availabilities'].append(gate_availability)

            sensor_health = {
                (sensor.gate_id, sensor.sensor_type): sensor.health_index
                for sensor in self.instance.sensors
            }
            self.results['sensor_healths'].append(sensor_health.copy())

            assert monitoring_cost <= (
                self.instance.monitoring_budget_per_hour
            )
            assert np.isclose(
                budget_utilization,
                monitoring_cost
                / self.instance.monitoring_budget_per_hour
                * 100,
            )

            if (hour + 1) % 12 == 0:
                print(
                    f'  Hour {hour + 1:2d}/'
                    f'{self.instance.time_periods}: '
                    f'gate availability = {reliability:.1f}%'
                )

        return self.results

In [ ]:
# Run the simulation and summarize the observed history.
pbsm = PriorityBasedSensorMonitoring(heuristic_instance)
results = pbsm.run_simulation()

final_reliability = np.mean(results['system_reliability'])
final_budget_utilization = np.mean(results['budget_utilization'])
total_vehicles = sum(
    gate.total_vehicles_processed for gate in heuristic_instance.gates
)

print('\nSIMULATION SUMMARY')
print(f'Mean gate availability: {final_reliability:.1f}%')
print(f'Mean budget utilization: {final_budget_utilization:.1f}%')
print(f'Total vehicles processed: {total_vehicles:,}')
print(f'Simulation duration: {heuristic_instance.time_periods} hours')

assert len(results['system_reliability']) == heuristic_instance.time_periods
assert len(results['monitoring_schedule']) == heuristic_instance.time_periods

The summary reports averages over the simulated hours, not a guaranteed long-run performance level. Because sensor failures use random draws and no seed is imposed, the exact values can vary between executions. Budget utilization measures the share of the hourly monitoring budget spent; it does not measure reliability or the value of the sensors selected.

In [ ]:
def analyze_monitoring_patterns(results, instance):
    """Summarize how often each sensor is selected and compare demand periods."""
    monitoring_counts = defaultdict(int)
    for hour_schedule in results['monitoring_schedule']:
        for gate_id, sensor_type in hour_schedule:
            monitoring_counts[(gate_id, sensor_type)] += 1

    total_hours = instance.time_periods
    monitoring_percentages = {}
    for sensor in instance.sensors:
        sensor_key = (sensor.gate_id, sensor.sensor_type)
        monitoring_percentages[sensor_key] = (
            monitoring_counts[sensor_key] / total_hours * 100
        )

    sorted_by_percentage = sorted(
        monitoring_percentages.items(),
        key=lambda item: item[1],
        reverse=True,
    )

    print('\nMONITORING PATTERN ANALYSIS')
    print('\nFive most frequently monitored sensors:')
    for (gate_id, sensor_type), percentage in sorted_by_percentage[:5]:
        print(
            f'  Gate {gate_id} - {sensor_type}: '
            f'{percentage:.1f}% of simulated hours'
        )

    print('\nThree least frequently monitored sensors:')
    for (gate_id, sensor_type), percentage in sorted_by_percentage[-3:]:
        print(
            f'  Gate {gate_id} - {sensor_type}: '
            f'{percentage:.1f}% of simulated hours'
        )

    peak_hours = {8, 9, 10, 17, 18, 19}
    peak_counts = []
    off_peak_counts = []

    for hour, schedule in enumerate(results['monitoring_schedule']):
        if hour % 24 in peak_hours:
            peak_counts.append(len(schedule))
        else:
            off_peak_counts.append(len(schedule))

    mean_peak = np.mean(peak_counts)
    mean_off_peak = np.mean(off_peak_counts)
    print('\nPeak versus off-peak periods')
    print(f'Mean sensors monitored in peak hours: {mean_peak:.1f}')
    print(f'Mean sensors monitored in other hours: {mean_off_peak:.1f}')
    print(f'Difference in means: {mean_peak - mean_off_peak:+.1f}')

    return monitoring_percentages, sorted_by_percentage


monitoring_percentages, sorted_sensors = analyze_monitoring_patterns(
    results,
    heuristic_instance,
)

Monitoring frequency is the percentage of simulated hours in which a sensor was selected. Sensors that were never selected are included with a frequency of zero. The peak-versus-other-hours comparison describes this example's schedule; it does not establish that the greedy rule always increases monitoring during peak demand. The budget can constrain the number of selections, and the priority ordering depends on all score components.

In [ ]:
def create_heuristic_visualizations(results, instance, monitoring_percentages):
    """Plot reliability, spending, monitoring frequency, and gate availability."""
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    fig.suptitle(
        'Priority-Based Sensor Monitoring Simulation',
        fontsize=16,
        fontweight='bold',
    )

    hours = list(range(len(results['system_reliability'])))

    reliability_axis = axes[0, 0]
    reliability = results['system_reliability']
    reliability_axis.plot(
        hours,
        reliability,
        linewidth=2,
        color='#2ecc71',
        label='Simulated gate availability',
    )
    reliability_axis.fill_between(
        hours,
        reliability,
        alpha=0.2,
        color='#2ecc71',
    )
    reliability_axis.set_title('Gate Availability Over Time')
    reliability_axis.set_xlabel('Simulation hour')
    reliability_axis.set_ylabel('Operational gates (%)')
    reliability_axis.set_ylim(0, 105)
    reliability_axis.grid(True, alpha=0.3)
    reliability_axis.legend()

    budget_axis = axes[0, 1]
    budget_axis.plot(
        hours,
        results['budget_utilization'],
        linewidth=2,
        color='#3498db',
        label='Budget spent',
    )
    budget_axis.axhline(
        y=100,
        color='red',
        linestyle='--',
        alpha=0.7,
        label='Hourly budget limit',
    )
    budget_axis.set_title('Hourly Budget Utilization')
    budget_axis.set_xlabel('Simulation hour')
    budget_axis.set_ylabel('Budget used (%)')
    budget_axis.set_ylim(0, 105)
    budget_axis.grid(True, alpha=0.3)
    budget_axis.legend()

    sensor_types = ['photo_eye', 'ground_loop', 'safety_beam']
    sensor_labels = ['Photo eye', 'Ground loop', 'Safety beam']
    gate_labels = [f'Gate {gate.gate_id}' for gate in instance.gates]
    monitoring_matrix = np.zeros((len(instance.gates), len(sensor_types)))

    for gate_index, gate in enumerate(instance.gates):
        for sensor_index, sensor_type in enumerate(sensor_types):
            monitoring_matrix[gate_index, sensor_index] = (
                monitoring_percentages[(gate.gate_id, sensor_type)]
            )

    heatmap_axis = axes[1, 0]
    heatmap = heatmap_axis.imshow(
        monitoring_matrix,
        cmap='RdYlGn',
        aspect='auto',
        vmin=0,
        vmax=100,
    )
    heatmap_axis.set_xticks(range(len(sensor_labels)))
    heatmap_axis.set_xticklabels(sensor_labels)
    heatmap_axis.set_yticks(range(len(gate_labels)))
    heatmap_axis.set_yticklabels(gate_labels)
    heatmap_axis.set_title('Monitoring Frequency by Sensor (%)')
    heatmap_axis.set_xlabel('Sensor type')
    heatmap_axis.set_ylabel('Gate')
    colorbar = fig.colorbar(heatmap, ax=heatmap_axis)
    colorbar.set_label('Simulated hours monitored (%)')

    for gate_index in range(len(instance.gates)):
        for sensor_index in range(len(sensor_types)):
            heatmap_axis.text(
                sensor_index,
                gate_index,
                f'{monitoring_matrix[gate_index, sensor_index]:.0f}',
                ha='center',
                va='center',
                color='black',
                fontweight='bold',
            )

    gate_availability = {
        gate.gate_id: [] for gate in instance.gates
    }
    for hour_data in results['gate_availabilities']:
        for gate_id, availability in hour_data.items():
            gate_availability[gate_id].append(availability)

    mean_availability = [
        np.mean(gate_availability[gate.gate_id])
        for gate in instance.gates
    ]
    availability_axis = axes[1, 1]
    bars = availability_axis.bar(
        gate_labels,
        mean_availability,
        color='#4c72b0',
    )
    availability_axis.set_title('Mean Availability by Gate')
    availability_axis.set_xlabel('Gate')
    availability_axis.set_ylabel('Operational hours (%)')
    availability_axis.set_ylim(0, 105)
    availability_axis.grid(axis='y', alpha=0.3)

    for bar, availability in zip(bars, mean_availability):
        availability_axis.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 1,
            f'{availability:.1f}%',
            ha='center',
            va='bottom',
            fontweight='bold',
        )

    fig.tight_layout()
    plt.show()
    return fig


fig = create_heuristic_visualizations(
    results,
    heuristic_instance,
    monitoring_percentages,
)

The panels show different quantities: gate availability is a percentage of gates operational in each hour, budget utilization is the percentage of the hourly spending limit used, and monitoring frequency is the percentage of hours each sensor was selected. These measures have different denominators and should not be interpreted as interchangeable performance scores.

In [ ]:
def performance_analysis(results, instance):
    """Report an operation-count proxy and retained-history size."""
    n_gates = len(instance.gates)
    n_sensors = len(instance.sensors)
    n_periods = instance.time_periods

    if n_sensors > 1:
        heap_factor = n_sensors * np.log2(n_sensors)
    else:
        heap_factor = n_sensors

    operation_proxy = n_periods * (
        2 * heap_factor + n_gates * n_sensors
    )

    print('\nCOMPUTATIONAL SCALING')
    print(
        'Per-hour time bound: '
        'O(S log S + G S), where G is the number of gates '
        'and S is the number of sensors.'
    )
    print(
        f'For this instance: G={n_gates}, S={n_sensors}, '
        f'T={n_periods} hours.'
    )
    print(
        'Operation-count proxy '
        '(heap work plus gate-sensor scans): '
        f'{operation_proxy:,.0f}'
    )
    print(
        'This proxy counts modeled operations; it is not a '
        'wall-clock runtime measurement.'
    )

    recorded_sensor_entries = sum(
        len(hour_data) for hour_data in results['sensor_healths']
    )
    recorded_gate_entries = sum(
        len(hour_data) for hour_data in results['gate_availabilities']
    )
    recorded_selections = sum(
        len(schedule) for schedule in results['monitoring_schedule']
    )

    print('\nRetained result records')
    print(f'Sensor-health entries: {recorded_sensor_entries:,}')
    print(f'Gate-availability entries: {recorded_gate_entries:,}')
    print(f'Selected-sensor entries: {recorded_selections:,}')
    print(
        'The retained histories grow with the number of simulated '
        'hours and sensors; Python object memory is not estimated '
        'from record counts.'
    )

    return operation_proxy


complexity_estimate = performance_analysis(results, heuristic_instance)

The operation-count proxy makes the dependence on gates, sensors, and time periods visible, but it is not a benchmark. In particular, it does not account for Python interpreter overhead, machine hardware, or plotting. The asymptotic bound describes how the algorithm's work grows as the instance becomes larger.

In [ ]:
def quality_gap_analysis(results, instance):
    """Report observed performance without assuming an optimal benchmark."""
    mean_gate_availability = float(
        np.mean(results['system_reliability'])
    )
    mean_budget_utilization = float(
        np.mean(results['budget_utilization'])
    )
    mean_hourly_cost = (
        mean_budget_utilization
        / 100
        * instance.monitoring_budget_per_hour
    )

    print('\nOBSERVED PERFORMANCE AND BENCHMARK SCOPE')
    print(
        'Mean simulated gate availability: '
        f'{mean_gate_availability:.1f}%'
    )
    print(
        'Mean hourly monitoring cost: '
        f'${mean_hourly_cost:.2f}'
    )
    print(
        'A quality gap to an optimum cannot be calculated from '
        'this simulation alone: no optimal solution or matching '
        'benchmark value is supplied here.'
    )
    print(
        'These observed metrics describe one stochastic run and '
        'do not establish optimality or a statistical guarantee.'
    )

    return {
        'mean_gate_availability': mean_gate_availability,
        'mean_budget_utilization': mean_budget_utilization,
        'mean_hourly_cost': mean_hourly_cost,
        'optimality_gap': None,
    }


quality_analysis = quality_gap_analysis(results, heuristic_instance)

## Interpretation and limitations

### What this heuristic demonstrates

- **Priority-based allocation:** Multiple sensor and gate characteristics contribute to a single score, allowing the algorithm to rank monitoring candidates.
- **Budget feasibility:** The selection rule does not spend more than the hourly budget. It can leave some budget unused when the remaining amount is smaller than the cost of every remaining sensor.
- **Dynamic decisions:** Health and failure history evolve during the simulation, while demand varies by hour. These changes can alter sensor priorities in later periods.
- **A transparent trade-off:** The model allocates limited monitoring capacity using a rule that is easy to inspect, but it does not evaluate every possible schedule or account for all future consequences.

### What the results do not establish

A greedy priority score is not an optimality certificate. No optimal schedule, comparable optimization result, or independently validated reliability target is computed in this notebook, so an optimality gap cannot be reported. Likewise, a single stochastic simulation does not establish a reliability guarantee. The failure probabilities, health transitions, gate threshold, and score weights are illustrative model assumptions rather than empirically calibrated quantities.

The reported computational bound is for the implementation shown: sensor scoring and heap selection, together with the gate-by-gate scans of sensors. The operation-count proxy is not a measured runtime and does not by itself demonstrate real-time performance at a particular facility scale.

### Scope for further study

A larger study could calibrate the state-transition assumptions using operational data, run repeated simulations to quantify variation, or add operational constraints such as maintenance capacity and sensor redundancy. Those additions would require explicit data and modeling choices; they are outside the present heuristic implementation.